# Evaluative Framing of Scholars in Historical Texts


#### This notebook presents a computational approach to examining how historians evaluate and frame scholars and other historical figures in their writings.

The llm assisted analysis identifies evaluative terms and expressions associated with named individuals and classifies their framing as positive, negative, neutral, or mixed. The workflow includes scholar identification, LLM-based evaluative framing analysis, extraction of structured results and normalization of personal names.

The resulting dataset supports the comparative study of evaluative representations in historical narratives.

In [ ]:
import pandas as pd
import numpy as np
import re
import string

#to be able to track progress when applying functions
from tqdm.notebook import tqdm
from tqdm.notebook import tqdm_notebook
tqdm_notebook.pandas()


In [18]:
# import the extracted quotes df
df = pd.read_csv('Kordatos_with_quotes.csv')

In [19]:
df.head(5)

,Chapters,text_monotonic_accents,text_No_Quotes,Λουθήρος_quote,Schmid_quote,Αριστοφάνης_quote,Ξενοφών_quote,Πτωχοπρόδρομος_quote,Γεωργιλάς_quote,Σοφιανός_quote,...,Οπαδοί_Μιστριώτη_quote,Δημητρακόπουλος_quote,Πρωτοδίκης_Στελλάκης_quote,Νιρβάνας_quote,num_wds,Total_book_words,Label,Class,clean_text_wd_count,Total_clean_text_words
0,ΠΡΟΛΟΓΟΣ,?Μή ζητάς στα λατινικά πως θα γράψεις καλά γερ...,ΛΟΥΘΗΡΟΣ Το γλωσσικό ζήτημα πάνε εκατό και παρ...,Μή ζητάς στα λατινικά πως θα γράψεις καλά γερμ...,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,879,58176,2,Kordatos,842,41172
1,ΕΙΣΑΓΩΓΗ,Η γλώσσα είναι ένα εργαλείο με το οποίο οι άνθ...,Η γλώσσα είναι ένα εργαλείο με το οποίο οι άνθ...,NaN,ο ελληνισμός των αυτοκρατορικών χρόνων δεν έχ...,"χρώμαι γάρ αυτού του στόματος τώ στρογγύλω, τ...",Έπειτα φωνήν πάσαν ακούοντες οι Αθηναίοι εξελ...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,4028,58176,2,Kordatos,3925,41172
2,ΚΕΦΑΛΑΙΟ ΠΡΩΤΟ,Το γλωσσικό ζήτημα στον καιρό της τουρκοκρατία...,Το γλωσσικό ζήτημα στον καιρό της τουρκοκρατία...,NaN,NaN,NaN,NaN,"Γείτοναν έχω κοσκινάν, φάρσωμα μάς χωρίζει κα...","Νάχεν αστράψει ο συρανός, νάχε καγεί η ώρα, Ήλ...","Και ας μη βαρυγκομήσει τινάς, αν εκείνα που ει...",...,NaN,NaN,NaN,NaN,17143,58176,2,Kordatos,10437,41172
3,ΚΕΦΑΛΑΙΟ ΔΕΥΤΕΡΟ,Το μεσουράνημα του αττικισμού Περίοδος 1821183...,Το μεσουράνημα του αττικισμού Περίοδος 1821183...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,4412,58176,2,Kordatos,3474,41172
4,ΚΕΦΑΛΑΙΟ ΤΡΙΤΟ,Το κήρυγμα του Ψυχάρη και η δύση του αττικισμο...,Το κήρυγμα του Ψυχάρη και η δύση του αττικισμο...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,5765,58176,2,Kordatos,4560,41172


In [20]:
print(df.columns.tolist())

['Chapters', 'text_monotonic_accents', 'text_No_Quotes', 'Λουθήρος_quote', 'Schmid_quote', 'Αριστοφάνης_quote', 'Ξενοφών_quote', 'Πτωχοπρόδρομος_quote', 'Γεωργιλάς_quote', 'Σοφιανός_quote', 'Κονταρής_quote', 'Μηνιάτης_quote', 'Κορνάρος_quote', 'Χορτάτζης_quote', 'Κύριλλος_Λούκαρης_quote', 'Βικέλας_quote', 'Αθανασίου_quote', 'Παπαδόπουλος_quote', 'Φιλαδελφεύς_quote', 'Δανιήλ_quote', 'Γεωργίου_quote', 'Κούμας_quote', 'Βλαντής_quote', 'Αθανάσιος_Πάριος_quote', 'Πέζαρος_quote', 'Βενιαμίν_quote', 'Συγγραφέας_Ελληνικής_Νομαρχίας_quote', 'Οικονόμος_quote', 'Ανώνυμος_quote', 'Ρήγας_Βελεστινλής_quote', 'Γρηγόριος_Ε_quote', 'Κοσμάς_Αιτωλός_quote', 'Κοραής_quote', 'Καταρτζής_quote', 'Κωνσταντάς_quote', 'Δανιήλ_Φιλιππίδης_quote', 'Χριστόπουλος_quote', 'Βηλαράς_quote', 'Ψαλλίδας_quote', 'Ψήφισμα_Σχολής_Κυδωνιών_quote', 'Θερειανός_quote', 'Κοδρικάς_quote', 'Νερουλός_quote', 'Νεόφυτος_Βάμβας_quote', 'Σολωμός_quote', 'Σύνθημα_λογιωτατισμού_quote', 'Σούτσος_quote', 'Ραγκαβής_quote', 'Τρικούπης_quote', 

In [21]:
# use the monotonic version of the full text and the authorial text
df = df[['Chapters', 'text_monotonic_accents', 'text_No_Quotes']]
df

,Chapters,text_monotonic_accents,text_No_Quotes
0,ΠΡΟΛΟΓΟΣ,?Μή ζητάς στα λατινικά πως θα γράψεις καλά γερ...,ΛΟΥΘΗΡΟΣ Το γλωσσικό ζήτημα πάνε εκατό και παρ...
1,ΕΙΣΑΓΩΓΗ,Η γλώσσα είναι ένα εργαλείο με το οποίο οι άνθ...,Η γλώσσα είναι ένα εργαλείο με το οποίο οι άνθ...
2,ΚΕΦΑΛΑΙΟ ΠΡΩΤΟ,Το γλωσσικό ζήτημα στον καιρό της τουρκοκρατία...,Το γλωσσικό ζήτημα στον καιρό της τουρκοκρατία...
3,ΚΕΦΑΛΑΙΟ ΔΕΥΤΕΡΟ,Το μεσουράνημα του αττικισμού Περίοδος 1821183...,Το μεσουράνημα του αττικισμού Περίοδος 1821183...
4,ΚΕΦΑΛΑΙΟ ΤΡΙΤΟ,Το κήρυγμα του Ψυχάρη και η δύση του αττικισμο...,Το κήρυγμα του Ψυχάρη και η δύση του αττικισμο...
5,ΚΕΦΑΛΑΙΟ ΤΕΤΑΡΤΟ,Το άπλωμα του δημοτικισμού. Δημοτική και φιλολ...,Το άπλωμα του δημοτικισμού. Δημοτική και φιλολ...
6,ΚΕΦΑΛΑΙΟ ΠΕΜΠΤΟ,"Τά αθεϊκά του Βόλου. Ο Βόλος, κέντρο πολιτικοκ...","Τά αθεϊκά του Βόλου. Ο Βόλος, κέντρο πολιτικοκ..."
7,ΚΕΦΑΛΑΙΟ ΕΚΤΟ,Η γλωσσοεκπαιδευτική μεταρρύθμιση. Η εισαγωγή ...,Η γλωσσοεκπαιδευτική μεταρρύθμιση. Η εισαγωγή ...
8,ΚΕΦΑΛΑΙΟ ΕΒΔΟΜΟ,Η κρίση του δημοτικισμού Τά αίτια. Δεν είναι κ...,Η κρίση του δημοτικισμού Τά αίτια. Δεν είναι κ...
9,ΚΕΦΑΛΑΙΟ ΟΓΔΟΟ,Νέων μεσαιωνισμών το ανάγνωσμα. Αντίδραση στη ...,Νέων μεσαιωνισμών το ανάγνωσμα. Αντίδραση στη ...


In [22]:
import re
import pandas as pd
import json

# Define the scholar patterns using regex
scholar_patterns = [
    r'Αιλιαν\w*', r'Άμαντ\w*', r'Ανανί\w*', r'Αναγνωστόπουλ\w*', r'Αναγνωστόπούλ\w*',
     r'Ανθρακίτ\w*', r'Ασώπι\w*', r'Ασωπί\w*', r'Βαλαωρίτ\w*', r'Βάρναλης',
    r'Βάμβα\w*', r'Βαρβαγιάννη\w*', r'Βαρθόλδης', r'Βασιλειάδ\w*', r'Βασιλόπουλ\w*', r'Βελεστινλής', r'Βελεστινλή',
    r'Βελών', r'Βελάστ\w*', r'Βενιζέλ\w*', r'Βενιαμίν', r'Βερναρδάκ\w*', r'Βηλαρ\w*', r'Βικέλα\w*',
    r'Βιλαρ\w*', r'Βιλλουασών', r'Βλάχο', r'Βλάχος', r'Βλάχου', r'Βλαστ\w*',
    r'Βολταίρ\w*', r'Βουλγάρεως', r'Βούλγαρη\w*', r'Βούλγαρις',
    r'Βραγκοβάν\w*', r'Γαζή\w*', r'Γερλάχι\w*', r'Γερλαχί\w*', r'Γεωργιλά\w*', r'Γκυς', r'Γαβριηλίδη\w*',
    r'Γλην\w*', r'Γορδι\w*', r'Πατριάρχη\w*', r'Δαμωδ\w*', r'Δελμούζ\w*',
    r'Δοσίθε\w*', r'Δούκα', r'Δούκας', r'Δουκάγγι\w*',
    r'Ελλάδι\w*', r'Ελλαδί\w*', r'Έρασμ\w*', r'Εφταλιώτ\w*', r'Ζαμπέλ\w*',
    r'Ζήνο\w*', r'Ζυγομαλ\w*', r'Ηλιάδ\w*', r'Θεοτόκ\w*', r'Ιλαρί\w*',
    r'Ιλδεβράνδ\w*', r'Καβαλιώτ\w*', r'Καβάσιλ\w*',
    r'Κακριδή\w*', r'Καλλεμβέργ\w*', r'Καπνί\w*', r'Καινιγ\w*',
    r'Καταρτζ\w*', r'Κοδρικ\w*', r'Κομμητ\w*', r'Κονδυλάκη*',
    r'Κονταρ\w*', r'Κόντ\w*', r'Κούμα\w*', r'Κοραή\w*', r'Κορνάρ\w*', r'Κριτόπουλ\w*',
    r'Κρομμύδ\w*', r'Κρούσ\w*', r'Κυμηνίτ\w*', r'Κωνσταντά\w*',
    r'Λάγγι\w*', r'Λάνδ\w*', r'Λάσκαρ\w*', r'Λορεντζάτο\w*', r'Λασκαράτο\w*', r'Λουδόλφ\w*', r'Λούκαρ\w*',r'Λουκάρ\w*',
    r'Μάξιμ\w*', r'Μαλάκ\w*', r'Μαργούνι\w*', r'Μεθόδι\w*',
    r'Μερκάδ\w*', r'Μηνιάτ\w*', r'Μιστριώτη\w*',
    r'Μοισιόδα\w*', r'Ναουσαί\w*', r'Νερουλ\w*', r'Νικηφόρ\w*', r'Νιρβάνα\w*',
    r'Νοταρ\w*', r'Ξανθόπουλο\w*', r'Ξανθουκίδ\w*', r'Οικονόμ\w*', r'Οστοβήκ', r'Παλαμά\w*',
    r'Πάλλ\w*', r'Παπαμόσχου', r'Παπαντωνίου', r'Παρθένι\w*', r'Παρισιν\w*',r'Πάριο\w*',
    r'Πηγά', r'Πηγάς', r'Πιακεντίν\w*', r'Πιανζόλ\w*', r'Πολυλά\w*', r'Πόρκι\w*', r'Πουότ\w*', r'Ραγκαβή\w*',
    r'Ραρτούρ\w*', r'Ροΐδη\w*', r'Ρουσάν\w*', r'Ρωμαν\w*', r'Σκούφο\w*', r'Σολωμός', r'Σολωμού', r'Σούτσο\w*',
    r'Σομμαβέρ\w*', r'Σοφιανός', r'Σοφιανού', r'Σπων', r'Στρατηγ\w*',
    r'Στουδιτ\w*', r'Σύλβι\w*', r'Τριανταφυλλιδ\w*', r'Τρικούπη\w*', r'Τριββεχόβ\w*', r'Τσούντα\w*', r'Τυπάλδο\w*',
    r'Φαρμακίδ\w*', r'Φίλελφ\w*', r'Φιλήντα\w*', r'Φωτιάδ\w*', r'Φιλιππίδ\w*', r'Χατζηδάκ\w*',
    r'Χριστόπουλ\w*', r'Χρυσόστομος', r'Χύχ\w*', r'Χατζής', r'Χατζή', r'Ψαλλίδα\w*', r'Ψυχάρ\w*',     r'Ξενοφών\w*',
    r'Ξενοφώντ\w*', r'Πτωχοπρόδρομ\w*', r'Χορτάτζ\w*', r'Αθανασί\w*', r'Παπαδόπουλ\w*', r'Φιλαδελφ\w*', r'Γεωργί\w*', r'Βλαντ\w*',
    r'Πέζαρ\w*', r'Θερειαν\w*', r'Τερτσέτ\w*', r'Σπηλιωτάκ\w*', r'Λιβαδ\w*', r'Χρυσοβέργ\w*', r'Ζουφρέ\w*',
    r'Καλιάφ\w*', r'Λογοθέτ\w*', r'Δημητρακόπουλ\w*', r'Στελλάκ\w*',
]




In [23]:
# Compile regex pattern
compiled_pattern = re.compile(r'\b(?:' + '|'.join(scholar_patterns) + r')\b')

def detect_scholars(text):
    """
    Detects scholars in a given text using predefined regex patterns.
    Returns a list of matched scholars.
    """
    matches = re.findall(compiled_pattern, text)
    return list(set(matches)) if matches else ["None"]  # Remove duplicates, return None if no match

In [24]:
# Apply scholar detection on the 'text_no_quotes' column
df['Detected_Scholars'] = df['text_No_Quotes'].progress_apply(detect_scholars)

  0%|          | 0/11 [00:00<?, ?it/s]

In [25]:
# Flatten the list of scholars and get unique values
unique_scholars = set(scholar for scholars_list in df['Detected_Scholars'].dropna() for scholar in scholars_list)
print(unique_scholars)

{'Ραγκαβήδων', 'Κακριδής', 'Λογοθέτη', 'Οικονόμος', 'Τυπάλδο', 'Στελλάκηδες', 'Μοισιόδαξ', 'Βασιλόπουλος', 'Κοραής', 'Βλάχος', 'Βάμβας', 'Θεοτόκη', 'Βλαστού', 'Βλαστό', 'Χρυσοβέργης', 'Ροΐδη', 'Κόντου', 'Σούτσο', 'Βερναρδάκη', 'Καλιάφα', 'Μιστριώτηδων', 'Βενιαμίν', 'Φιλήντα', 'Τρικούπη', 'Δημητρακόπουλος', 'Φιλήντας', 'Βηλαράς', 'Παπαντωνίου', 'Χρυσόστομος', 'Χριστόπουλος', 'Πέζαρος', 'Ψαλλίδας', 'Μαργούνιος', 'Γεωργιλάς', 'Τρικούπης', 'Βάρναλης', 'Ραγκαβή', 'Φωτιάδης', 'Τσούντα', 'Σκούφο', 'Ψυχάρη', 'Γληνό', 'Εφταλιώτης', 'Σοφιανού', 'Λασκαράτος', 'Ψυχάρης', 'Καταρτζής', 'Πάριος', 'Κονταρής', 'Παπαμόσχου', 'Μηνιάτης', 'Ξενοφώντα', 'Βικέλας', 'Βούλγαρη', 'Σπηλιωτάκης', 'Λογοθέτες', 'Μάξιμος', 'Κοδρικά', 'Σκούφου', 'Στελλάκης', 'Γαζής', 'Οικονόμων', 'Χατζή', 'Νερουλός', 'Δαμωδός', 'Ραγκαβής', 'Γαβριηλίδη', 'Μεθόδιος', 'Λούκαρης', 'Ιλαρίωνα', 'Δημητρακόπουλο', 'Βαρβαγιάννη', 'Μιστριώτη', 'Παπαδόπουλος', 'Δελμούζος', 'Πτωχοπρόδρομο', 'Βλαστός', 'Φωτιάδη', 'Πηγάς', 'Κωνσταντάς', 'Κόντος', 

In [26]:
#display the texts better
pd.set_option('display.width', 50)
pd.set_option('display.max_colwidth', 50)

In [27]:
df

,Chapters,text_monotonic_accents,text_No_Quotes,Detected_Scholars
0,ΠΡΟΛΟΓΟΣ,?Μή ζητάς στα λατινικά πως θα γράψεις καλά γερ...,ΛΟΥΘΗΡΟΣ Το γλωσσικό ζήτημα πάνε εκατό και παρ...,"[Κοραή, Μιστριώτηδες, Κοραής, Χατζηδάκηδες, Μι..."
1,ΕΙΣΑΓΩΓΗ,Η γλώσσα είναι ένα εργαλείο με το οποίο οι άνθ...,Η γλώσσα είναι ένα εργαλείο με το οποίο οι άνθ...,"[Χρυσόστομος, Ροΐδης, Ξενοφώντα]"
2,ΚΕΦΑΛΑΙΟ ΠΡΩΤΟ,Το γλωσσικό ζήτημα στον καιρό της τουρκοκρατία...,Το γλωσσικό ζήτημα στον καιρό της τουρκοκρατία...,"[Σοφιανός, Κονταρής, Οικονόμος, Ραρτούρος, Κού..."
3,ΚΕΦΑΛΑΙΟ ΔΕΥΤΕΡΟ,Το μεσουράνημα του αττικισμού Περίοδος 1821183...,Το μεσουράνημα του αττικισμού Περίοδος 1821183...,"[Ραγκαβήδων, Βαλαωρίτης, Τυπάλδο, Χριστόπουλο,..."
4,ΚΕΦΑΛΑΙΟ ΤΡΙΤΟ,Το κήρυγμα του Ψυχάρη και η δύση του αττικισμο...,Το κήρυγμα του Ψυχάρη και η δύση του αττικισμο...,"[Γαβριηλίδης, Λιβαδάς, Κοραής, Χατζηδάκη, Θεοτ..."
5,ΚΕΦΑΛΑΙΟ ΤΕΤΑΡΤΟ,Το άπλωμα του δημοτικισμού. Δημοτική και φιλολ...,Το άπλωμα του δημοτικισμού. Δημοτική και φιλολ...,"[Παλαμάς, Βενιζέλος, Χατζηδάκη, Νιρβάνας, Κορα..."
6,ΚΕΦΑΛΑΙΟ ΠΕΜΠΤΟ,"Τά αθεϊκά του Βόλου. Ο Βόλος, κέντρο πολιτικοκ...","Τά αθεϊκά του Βόλου. Ο Βόλος, κέντρο πολιτικοκ...","[Δελμούζο, Βάρναλης, Δελμούζος, Γληνός, Πάλλη,..."
7,ΚΕΦΑΛΑΙΟ ΕΚΤΟ,Η γλωσσοεκπαιδευτική μεταρρύθμιση. Η εισαγωγή ...,Η γλωσσοεκπαιδευτική μεταρρύθμιση. Η εισαγωγή ...,"[Οικονόμου, Βενιζέλος, Χατζηδάκη, Δελμούζου, Β..."
8,ΚΕΦΑΛΑΙΟ ΕΒΔΟΜΟ,Η κρίση του δημοτικισμού Τά αίτια. Δεν είναι κ...,Η κρίση του δημοτικισμού Τά αίτια. Δεν είναι κ...,"[Παλαμά, Βλαστού, Μιστριώτηδες, Παπαμόσχου, Βλ..."
9,ΚΕΦΑΛΑΙΟ ΟΓΔΟΟ,Νέων μεσαιωνισμών το ανάγνωσμα. Αντίδραση στη ...,Νέων μεσαιωνισμών το ανάγνωσμα. Αντίδραση στη ...,"[Κακριδής, Λογοθέτη, Δελμούζο, Δελμούζος, Τσού..."


### Μake test dfs

In [28]:
test0_1 = df[0:2] # preface and Introduction
test1 = df[2:3] #chap 1
test2 = df[3:4] #chap 2
test3 = df[4:5] #....
test4 = df[5:6]
test5 = df[6:7]
test6 = df[7:8]
test7 = df[8:9]
test8 = df[9:10]
test9 = df[10:]

In [29]:
pd.set_option('display.width', 50)
pd.set_option('display.max_colwidth', 150)

In [30]:
test5

,Chapters,text_monotonic_accents,text_No_Quotes,Detected_Scholars
6,ΚΕΦΑΛΑΙΟ ΠΕΜΠΤΟ,"Τά αθεϊκά του Βόλου. Ο Βόλος, κέντρο πολιτικοκοινωνικών ζυμώσεων στα χρόνια 19051910 Ο Βόλος από τη γεωγραφική του θέση είχε γίνει σπουδαίο εμπορο...","Τά αθεϊκά του Βόλου. Ο Βόλος, κέντρο πολιτικοκοινωνικών ζυμώσεων στα χρόνια 19051910 Ο Βόλος από τη γεωγραφική του θέση είχε γίνει σπουδαίο εμπορο...","[Δελμούζο, Βάρναλης, Δελμούζος, Γληνός, Πάλλη, Τσούντας, Σολωμού, Δελμούζου, Μιστριώτη]"


In [ ]:
#display the texts better
pd.set_option('display.width', 50)
pd.set_option('display.max_colwidth', 100)

# LLM impelentation for framing detection

In [125]:
import json
from openai import OpenAI

client = OpenAI()  

In [204]:
def analyze_framing(text, Detected_Scholars, model="gpt-4.1"):

    scholar_names = ", ".join(Detected_Scholars)  # Format scholar names for clarity    # Debugging: Print the scholar_names inside the function
     
    if not Detected_Scholars:
        return '{"logioi": []}'       
    
    prompt = f"""
    Είσαι  ένας αναλυτής λογου και αναλύεις πως ο συγγραφέας του δοθέντος κειμένου "{text}" αξιολογεί και πλαισιώνει συγκεκριμένους λογίους και άλλες προσωπικότητες που αναφέρει στο κείμενοt.

    Δοθέντος του ακόλουθου κειμένου:
    "{text}"

    1. Οι λογίοι που εμφανίζονται είναι μόνο αυτοί στη λίστα: {scholar_names}. Μην αναφέρεις άλλους λογίους που δεν περιλαμβάνονται στη λίστα.
    2. Προσδιόρισε τους αξιολογικούς όρους (θετικούς ή αρνητικούς) που χρησιμοποιούνται για να τους περιγράψουν.
    3. Προσδιόρισε φράσεις που χρησιμοποιούνται με αξιολογικό τρόπο (π.χ. 'επιμόνως εδόξαζεν', 'είχε την παράδοξη ιδέα')
    4. Αποφάσισε αν η πλαισίωση κάθε λογίου είναι «θετική», «αρνητική», «ουδέτερη» ή «μικτή». Χρησιμοποίησε «μικτή» όταν συνυπάρχουν σαφείς θετικές και αρνητικές αξιολογήσεις του ίδιου λογίου. Χρησιμοποίησε «ουδέτερη» όταν δεν διαπιστώνεται σαφής θετική ή αρνητική αξιολόγηση.
    5. Ταξινόμησε  την αξιολογική πλαισίωση του κάθε ατόμου στη λίστα, όχι απλά τις γνώμες ή τις στάσεις που εκφράζει το αναφερόμενο άτομο προς άλλα πρόσωπα. Να διακρίνεις ξεκάθαρα ανάμεσα στην αξιολόγηση του ίδιου του συγγραφέα του κειμένου και τις αξιολογήσεις που αφορούν τρίτα πρόσωπα.
    6. Δωσε με μια πρόταση την αιτιολόγηση για την απόφαση σου. 

    Επίστρεψε αποκλειστικά έγκυρο JSON με την ακόλουθη δομή:
    {{
  "logioi": [
    {{
      "onoma": "Όνομα λογίου",
      "mentions": [
        {{
          "axiologikoi_oroi": [],
          "fraseis": [],
          "plaisiosi": "θετική",
          "aitiologisi": "Σύντομη αιτιολόγηση."
        }}
      ]
    }}
  ]
}}

Οι επιτρεπόμενες τιμές του "plaisiosi" είναι:
"θετική", "αρνητική", "ουδέτερη", "μικτή".

Αν δεν εντοπίζεται κανένας από τους λογίους της λίστας, επιστρέψτε {{"logioi": []}}.
"""

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "user", "content": prompt}
            ],
            temperature=0,
            response_format={"type": "json_object"}
        )

        json_output = response.choices[0].message.content
        return json_output

    except Exception as e:
        print(f"Σφάλμα κατά την κλήση του OpenAI API: {e}")
        return None

In [219]:
from tqdm.auto import tqdm
tqdm.pandas()

test9["Framing_Analysis"] = test9.progress_apply(
    lambda row: analyze_framing(
        row["text_No_Quotes"],
        row["Detected_Scholars"]
    )
    if isinstance(row["Detected_Scholars"], list) and row["Detected_Scholars"]
    else '{"logioi": []}',
    axis=1
)

  0%|          | 0/1 [00:00<?, ?it/s]

In [220]:
# run evaluative framing analysis only for unprocessed chapters
# save a checkpoint after each iteration to allow resuming interrupted API processing without repeating completed calls 

# if there is no column 
if "Framing_Analysis" not in test9.columns:
    test9["Framing_Analysis"] = None

# go over all the rows of the chapter
for idx, row in tqdm(test9.iterrows(), total=len(test9)):

    if pd.notna(row["Framing_Analysis"]):
        continue

    scholars = row["Detected_Scholars"]

    if isinstance(scholars, list) and scholars:
        result = analyze_framing(
            row["text_No_Quotes"],
            scholars
        )
    else:
        result = '{"logioi": []}'

    test9.at[idx, "Framing_Analysis"] = result

    # create a checkpoint for each row
    test9.to_pickle("test9_framing_checkpoint.pkl")

  0%|          | 0/1 [00:00<?, ?it/s]

In [221]:
def make_framing_df(df):
    records = []

    for _, row in df.iterrows():

        if not isinstance(row["Framing_Analysis"], str):
            continue

        try:
            data = json.loads(row["Framing_Analysis"])
        except json.JSONDecodeError:
            continue

        for scholar in data.get("logioi", []):
            records.append({
                "Chapter": row["Chapters"],
                "Scholar": scholar["onoma"],
                "mentions": scholar.get("mentions", [])
            })

    if not records:
        return pd.DataFrame()

    framing_df = pd.json_normalize(
        records,
        record_path="mentions",
        meta=["Chapter", "Scholar"]
    )

    return framing_df.rename(columns={
        "axiologikoi_oroi": "Axiologikoi_Oroi",
        "fraseis": "Fraseis",
        "plaisiosi": "Plaisiosi",
        "aitiologisi": "Aitiologisi"
    })

In [116]:
#display the texts better
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

### Manual checks

In [117]:
#framing_df0_1 = make_framing_df(test0_1)
#framing_df0_1.tail(10)

In [124]:
#framing_df1 = make_framing_df(test1)
#framing_df1.head(10)

In [161]:
#framing_df2= make_framing_df(test2)
#framing_df2.head(10)

In [158]:
framing_df3= make_framing_df(test3)
framing_df3.head(10)

,Axiologikoi_Oroi,Fraseis,Plaisiosi,Aitiologisi,Chapter,Scholar
0,"[μυαλό στείρο, αντίδραση, φέρνουν σύγχυση, ποτίσουν με το φαρμάκι της αντίδρασης, Ταμπού, στειρα και άγονη, σπαταλούσε την αρχαιομάθεια, διορθώσει...","[Βλέποντας οι αττικιστές πως το γλωσσικό τους κάστρο γκρεμίζονταν από τα βόλια της κριτικής του Βερναρδάκη, συσπειρώθηκαν γύρω στον Κόντο και με φ...",αρνητική,"Ο συγγραφέας χρησιμοποιεί σαφώς αρνητικούς όρους και ειρωνικές φράσεις για να περιγράψει τον Κόντο, τονίζοντας τη στείρα, αντιδραστική και αναποτε...",ΚΕΦΑΛΑΙΟ ΤΡΙΤΟ,Κόντος
1,[παραδείγματα του Σκούφου],"[πολύ ζημιωθήκαμε, με το να μην επικρατήσει στο γραφτό λόγο το παράδειγμα της κρητικής ποίησης, καθώς και οι συστάσεις και τα παραδείγματα του Σκο...",θετική,"Ο συγγραφέας εκφράζει θετική αξιολόγηση για το παράδειγμα του Σκούφου, θεωρώντας ζημιά που δεν επικράτησε το πρότυπό του.",ΚΕΦΑΛΑΙΟ ΤΡΙΤΟ,Σκούφου
2,"[πατριώτης, αερολογία, αρχηγός των καθαρευουσιάνων, πολεμική είχε ξεπεράσει κάθε δριο ευπρέπειας, φανατισμός ήταν ο μόνος οδηγός]","[Ήταν πατριώτης ως το κόκκαλο, όπως πατριώτες ήταν και ο Μιστριώτης και ο Κόντος., Η πολιτική αυτή αερολογία έκανε τους μεγαλοαστούς του εξωτερικο...",αρνητική,"Παρά τη σύντομη αναφορά ως 'πατριώτης', η συνολική πλαισίωση είναι αρνητική λόγω της σύνδεσης με αερολογία, φανατισμό και αντιδραστική στάση.",ΚΕΦΑΛΑΙΟ ΤΡΙΤΟ,Μιστριώτης
3,[παραδείγματα του Μηνιάτη],"[πολύ ζημιωθήκαμε, με το να μην επικρατήσει στο γραφτό λόγο το παράδειγμα της κρητικής ποίησης, καθώς και οι συστάσεις και τα παραδείγματα του Σκο...",θετική,"Ο συγγραφέας θεωρεί θετικό το παράδειγμα του Μηνιάτη, εκφράζοντας λύπη που δεν επικράτησε.",ΚΕΦΑΛΑΙΟ ΤΡΙΤΟ,Μηνιάτη
4,"[πατριώτης ως το κόκκαλο, εθνικό έργο, σωβινισμός του Ψυχάρη είναι ολοφάνερος, χτυπητός, ωμός, αντικειμενικά, ως γλωσσικό κίνημα, ωφέλησε, επαναστ...","[Το Ταξίδι του σημειώνει ένα σταθμό στην ιστορία του γλωσσικού ζητήματος κι είναι μαζί το ευαγγέλιο του νεώτερου δημοτικισμού., Ο Ψυχάρης λοιπόν δ...",μικτή,"Ο συγγραφέας αναγνωρίζει τη θετική συμβολή του Ψυχάρη στο γλωσσικό ζήτημα, αλλά τον επικρίνει έντονα για εθνικισμό, σωβινισμό και αντιδραστική κοι...",ΚΕΦΑΛΑΙΟ ΤΡΙΤΟ,Ψυχάρης
5,"[οι γνώμες του Κοραή ήταν οι πιο σωστές, δεν έκανε καλό, εμπόδισε, με το κύρος και τα γραψίματά του, να καθιερωθεί στον πεζό λόγο και στην ποίηση ...",[Ηταν πολλοί τώρα που δεν πίστευαν πως ήταν δυνατό να ξαναγυρίσουμε στο γλωσσικό καθεστώς της αρχαίας και παραδέχονταν πως οι γνώμες του Κοραή ήτα...,μικτή,"Αρχικά αναγνωρίζεται θετικά η ορθότητα των απόψεών του, αλλά στη συνέχεια του αποδίδεται αρνητική επίδραση στο γλωσσικό ζήτημα.",ΚΕΦΑΛΑΙΟ ΤΡΙΤΟ,Κοραή
6,"[αναγνώρισε τον κατήφορο που πήραν οι αρχαϊστές, υποστηριχτής της δημοτικής, δεν έπαψε για πολλά χρόνια να είναι ο υποστηριχτής της δημοτικής, δεν...","[Κι αυτός ακόμα οΓ Χατζηδάκης εξεγέρθηκε και αναγνώρισε τον κατήφορο που πήραν οι αρχαϊστές., δεν έπαψε για πολλά χρόνια να είναι ο υποστηριχτής τ...",αρνητική,"Παρά την αρχική αναγνώριση, η συνολική πλαισίωση είναι έντονα αρνητική, με έμφαση στην υποκρισία, την αντίδραση και τη συκοφαντία.",ΚΕΦΑΛΑΙΟ ΤΡΙΤΟ,Χατζηδάκης
7,"[φιλόλογος και λογοτέχνης, εξαιρετική φυσιογνωμία, γερός φιλόλογος και λογοτέχνης με όνομα, πολλά πνευματικά χαρίσματα, πολυδιαβασμένος, γερή ιστο...","[Ο Βερναρδάκης ήταν μια εξαιρετική φυσιογνωμία του καιρού του., Καταλάβαινε λοιπόν πως μια από τις αιτίες που έφεραν το πνευματικό αργοπόρημα του ...",θετική,"Ο συγγραφέας εκφράζει έντονα θετική στάση για τον Βερναρδάκη, τονίζοντας την πνευματική του αξία, την επιστημονικότητα και την αποτελεσματικότητα ...",ΚΕΦΑΛΑΙΟ ΤΡΙΤΟ,Βερναρδάκης
8,"[πιο ακλόνητος πολεμιστής και στυλοβάτης του ορθόδοξου ψυχαρισμού, μεγάλο γεγονός, μέγα σκάνδαλο, σελίδες που ζωντανεύουν τον Ομηρο, δεν είναι απ ...","[Ο Αλέξ. Πάλλης, διευθυντής πολλά χρόνια του εμπορικού Οίκου Ράλλη, στάθηκε για χρόνια ο πιο ακλόνητος πολεμιστής και στυλοβάτης του ορθόδοξου ψυχ...",μικτή,"Ο συγγραφέας αναγνωρίζει τη συμβολή του Πάλλη στη γλωσσική μεταρρύθμιση, αλλά ασκεί και ειρωνική/κριτική στάση για τ

In [167]:
#framing_df4= make_framing_df(test4)
#framing_df4.tail(10)

In [222]:
#framing_df5= make_framing_df(test5)
#framing_df5.tail(10)

In [217]:
#framing_df6= make_framing_df(test6)
#framing_df6.tail(10)

In [223]:
#framing_df7= make_framing_df(test7)
#framing_df7.head(10)

In [103]:
#framing_df8 = make_framing_df(test8)
#framing_df8

In [31]:
#framing_df9 = make_framing_df(test9)
#framing_df9.head(10)

### Concatenate

In [274]:
combined_df = pd.concat([framing_df0_1, framing_df1, framing_df2, framing_df3, framing_df4, framing_df5, framing_df6, framing_df7, framing_df8, framing_df9], ignore_index=True)

In [275]:
combined_df.head(4)

,Axiologikoi_Oroi,Fraseis,Plaisiosi,Aitiologisi,Chapter,Scholar
0,"[πατριδοκαπηλία, μεγάλοι του Πανεπιστημίου μας δασκάλοι, πιστή αντιγραφή, κατάντημα]","[Ανοίξτε την εισήγηση του Μιστριώτη στο Σεβαστοπούλειο διαγωνισμό 1908 για την καλλιτέρευσιν και προστασίαν της γλώσσης, για να δείτε πως ο παρακείμενος, οι δοτικές, οι μέσοι αόριστοι και οι υπερσυντέλικοι μαζί με την ευκτική είναι σωστά τηλεβόλα., Σ αυτό το κατάντημα της πατριδοκαπηλίας φτάσανε οι μεγάλοι του Πανεπιστημίου μας δασκάλοι., Κι ο λογιωτατισμός είχε κι αυτός τους δικούς του Χατζηδάκηδες, Μιστριώτηδες. Πιστή αντιγραφή μάλιστα.]",αρνητική,"Ο συγγραφέας χρησιμοποιεί ειρωνικές και απαξιωτικές εκφράσεις για τον Μιστριώτη, τονίζοντας την πατριδοκαπηλία, την υπερβολή και την έλλειψη πρωτοτυπίας.",ΠΡΟΛΟΓΟΣ,Μιστριώτη
1,[πιστή αντιγραφή],"[Κι ο λογιωτατισμός είχε κι αυτός τους δικούς του Χατζηδάκηδες, Μιστριώτηδες. Πιστή αντιγραφή μάλιστα.]",αρνητική,"Η αναφορά στους 'Μιστριώτηδες' γίνεται με ειρωνικό και απαξιωτικό τρόπο, υποδηλώνοντας έλλειψη πρωτοτυπίας και αρνητική στάση.",ΠΡΟΛΟΓΟΣ,Μιστριώτηδες
2,"[δε θα περνούσε κι αυτός για, όπως τονε κατηγόρησαν οι Φαναριώτες λογιώτατοι της εποχής του, μαζί με το Πατριαρχείο]","[Αν ήταν, όπως λένε πολλοί, ζήτημα διαφωτισμού της κοινής γνώμης, μού φαίνεται πως ένας Κοραής δε θα περνούσε κι αυτός για, όπως τονε κατηγόρησαν οι Φαναριώτες λογιώτατοι της εποχής του, μαζί με το Πατριαρχείο.]",θετική,"Ο συγγραφέας παρουσιάζει τον Κοραή ως θύμα άδικης κατηγορίας και ως παράδειγμα διαφωτιστή, υπονοώντας θετική στάση προς το πρόσωπό του.",ΠΡΟΛΟΓΟΣ,Κοραής
3,"[δε θα περνούσε κι αυτός για, όπως τονε κατηγόρησαν οι Φαναριώτες λογιώτατοι της εποχής του, μαζί με το Πατριαρχείο]","[Αν ήταν, όπως λένε πολλοί, ζήτημα διαφωτισμού της κοινής γνώμης, μού φαίνεται πως ένας Κοραής δε θα περνούσε κι αυτός για, όπως τονε κατηγόρησαν οι Φαναριώτες λογιώτατοι της εποχής του, μαζί με το Πατριαρχείο.]",θετική,"Η αναφορά στον Κοραή είναι θετική, καθώς παρουσιάζεται ως διαφωτιστής που αδίκως κατηγορήθηκε από τους συντηρητικούς της εποχής του.",ΠΡΟΛΟΓΟΣ,Κοραή


In [276]:
combined_df.columns

Index(['Axiologikoi_Oroi', 'Fraseis', 'Plaisiosi', 'Aitiologisi', 'Chapter', 'Scholar'], dtype='str')

In [277]:
combined_df = combined_df [['Chapter', 'Scholar', 'Axiologikoi_Oroi', 'Fraseis',
       'Plaisiosi', 'Aitiologisi']]

In [278]:
name_mapping = {
    "Φιλιππίδη": "Φιλιππίδης",
    "Γληνού": "Γληνός",
    "Γληνό": "Γληνός",
    "Κακριδή": "Κακριδής",
    "Κοδρικά": "Κοδρικάς",
    "Φιλήντα": "Φιλήντας",
    "Σκούφου": "Σκούφος",
    "Σκούφο": "Σκούφος",
    "Λούκαρη": "Λούκαρης",
    "Ροΐδη": "Ροΐδης",
    "Πάλλη": "Πάλλης",
    "Βλαστό": "Βλαστός",
    "Βλαστού": "Βλαστός",
    "Ψαλλίδα": "Ψαλλίδας",
    "Παλαμά": "Παλαμάς",
    "Δελμούζο": "Δελμούζος",
    "Δελμούζου": "Δελμούζος",
    "Βούλγαρη": "Βούλγαρης",
    "Σοφιανού": "Σοφιανός",
    "Τυπάλδο": "Τυπάλδος",
    "Τρικούπη": "Τρικούπης",
    "Ψυχάρη": "Ψυχάρης",
    "Καταρτζή": "Καταρτζής",
    "Κοραή": "Κοραής",
    "Γαβριηλίδη": "Γαβριηλίδης",
    "Βηλαρά": "Βηλαράς",
    "Κωνσταντά": "Κωνσταντάς",
    "Σούτσο": "Σούτσος",
    "Σούτσου": "Σούτσος",
    "Βαλαωρίτη": "Βαλαωρίτης",
    "Βενιζέλου": "Βενιζέλος",
    "Βενιζέλο": "Βενιζέλος",
    "Τσούντα": "Τσούντας",
    "Καβαλιώτη": "Καβαλιώτης",
    "Βαρβαγιάννη": "Βαρβαγιάννης",
    "Γαζή": "Γαζής",
    "Μιστριώτη": "Μιστριώτης",
    "Χατζηδάκη": "Χατζηδάκης",
    "Κονδυλάκη": "Κονδυλάκης",
    "Κούμα": "Κούμας",
    "Μάξιμο": "Μάξιμος",
    "Οικονόμου": "Οικονόμος",
    "Πολυλά": "Πολυλάς",
    "Μηνιάτη": "Μηνιάτης",
    "Ραγκαβή": "Ραγκαβής",
    "Κόντου": "Κόντος",
    "Κόντο": "Κόντος",
    "Λασκαράτο": "Λασκαράτος",
    "Κομμητά": "Κομμητάς",
    "Χριστόπουλο": "Χριστόπουλος",
    "Βερναρδάκη": "Βερναρδάκης",
    "Φωτιάδη": "Φωτιάδης",
    "Σολωμού": "Σολωμός",
    "Βελεστινλή": "Βελεστινλής",
    "Ασωπίου": "Ασώπιος",
    "Χατζή": "Χατζής",
    "Δημητρακόπουλο": "Δημητρακόπουλος",
    "Λογοθέτη": "Λογοθέτης",
    "Παπαδόπουλο": "Παπαδόπουλος",
    "Οικονόμων": "Οικονόμος",
    "Πτωχοπρόδρομου": "Πτωχοπρόδρομος",
    "Πτωχοπρόδρομο": "Πτωχοπρόδρομος",
    "Πτωχοπρόδρομων": "Πτωχοπρόδρομος",
    "Τερτσέτη": "Τερτσέτης",
    "Γεωργίου": "Γεώργιος",
    "Ξενοφώντα": "Ξενοφών",
    "Ιλαρίωνα": "Ιλαρίων",
    "Φαρμακίδη": "Φαρμακίδης",
}

In [279]:
combined_df["Scholars_Norm"] = (
    combined_df["Scholar"]
    .replace(name_mapping)
)

In [280]:
combined_df.head(14)

,Chapter,Scholar,Axiologikoi_Oroi,Fraseis,Plaisiosi,Aitiologisi,Scholars_Norm
0,ΠΡΟΛΟΓΟΣ,Μιστριώτη,"[πατριδοκαπηλία, μεγάλοι του Πανεπιστημίου μας δασκάλοι, πιστή αντιγραφή, κατάντημα]","[Ανοίξτε την εισήγηση του Μιστριώτη στο Σεβαστοπούλειο διαγωνισμό 1908 για την καλλιτέρευσιν και προστασίαν της γλώσσης, για να δείτε πως ο παρακείμενος, οι δοτικές, οι μέσοι αόριστοι και οι υπερσυντέλικοι μαζί με την ευκτική είναι σωστά τηλεβόλα., Σ αυτό το κατάντημα της πατριδοκαπηλίας φτάσανε οι μεγάλοι του Πανεπιστημίου μας δασκάλοι., Κι ο λογιωτατισμός είχε κι αυτός τους δικούς του Χατζηδάκηδες, Μιστριώτηδες. Πιστή αντιγραφή μάλιστα.]",αρνητική,"Ο συγγραφέας χρησιμοποιεί ειρωνικές και απαξιωτικές εκφράσεις για τον Μιστριώτη, τονίζοντας την πατριδοκαπηλία, την υπερβολή και την έλλειψη πρωτοτυπίας.",Μιστριώτης
1,ΠΡΟΛΟΓΟΣ,Μιστριώτηδες,[πιστή αντιγραφή],"[Κι ο λογιωτατισμός είχε κι αυτός τους δικούς του Χατζηδάκηδες, Μιστριώτηδες. Πιστή αντιγραφή μάλιστα.]",αρνητική,"Η αναφορά στους 'Μιστριώτηδες' γίνεται με ειρωνικό και απαξιωτικό τρόπο, υποδηλώνοντας έλλειψη πρωτοτυπίας και αρνητική στάση.",Μιστριώτηδες
2,ΠΡΟΛΟΓΟΣ,Κοραής,"[δε θα περνούσε κι αυτός για, όπως τονε κατηγόρησαν οι Φαναριώτες λογιώτατοι της εποχής του, μαζί με το Πατριαρχείο]","[Αν ήταν, όπως λένε πολλοί, ζήτημα διαφωτισμού της κοινής γνώμης, μού φαίνεται πως ένας Κοραής δε θα περνούσε κι αυτός για, όπως τονε κατηγόρησαν οι Φαναριώτες λογιώτατοι της εποχής του, μαζί με το Πατριαρχείο.]",θετική,"Ο συγγραφέας παρουσιάζει τον Κοραή ως θύμα άδικης κατηγορίας και ως παράδειγμα διαφωτιστή, υπονοώντας θετική στάση προς το πρόσωπό του.",Κοραής
3,ΠΡΟΛΟΓΟΣ,Κοραή,"[δε θα περνούσε κι αυτός για, όπως τονε κατηγόρησαν οι Φαναριώτες λογιώτατοι της εποχής του, μαζί με το Πατριαρχείο]","[Αν ήταν, όπως λένε πολλοί, ζήτημα διαφωτισμού της κοινής γνώμης, μού φαίνεται πως ένας Κοραής δε θα περνούσε κι αυτός για, όπως τονε κατηγόρησαν οι Φαναριώτες λογιώτατοι της εποχής του, μαζί με το Πατριαρχείο.]",θετική,"Η αναφορά στον Κοραή είναι θετική, καθώς παρουσιάζεται ως διαφωτιστής που αδίκως κατηγορήθηκε από τους συντηρητικούς της εποχής του.",Κοραής
4,ΠΡΟΛΟΓΟΣ,Χατζηδάκηδες,[πιστή αντιγραφή],"[Κι ο λογιωτατισμός είχε κι αυτός τους δικούς του Χατζηδάκηδες, Μιστριώτηδες. Πιστή αντιγραφή μάλιστα.]",αρνητική,"Η αναφορά στους 'Χατζηδάκηδες' γίνεται ειρωνικά και απαξιωτικά, ως μέρος μιας παράδοσης μιμητισμού και συντηρητισμού.",Χατζηδάκηδες
5,ΕΙΣΑΓΩΓΗ,Ροΐδης,[καλός πεζογράφος],"[με το να γράψει στην καθαρεύουσα, ξεχάστηκε και από τις μέλλουσες γενιές δε θα διαβάζεται καθόλου]",μικτή,"Ο συγγραφέας αναγνωρίζει το ταλέντο του Ροΐδη ως πεζογράφου (θετική αξιολόγηση), αλλά ταυτόχρονα τον κρίνει αρνητικά επειδή έγραψε στην καθαρεύουσα, με αποτέλεσμα να ξεχαστεί (αρνητική αξιολόγηση).",Ροΐδης
6,ΕΙΣΑΓΩΓΗ,Χρυσόστομος,[],[],ουδέτερη,"Ο Ιωάννης ο Χρυσόστομος αναφέρεται απλώς ως ένας από τους πατέρες της εκκλησίας που χρησιμοποίησαν τη λογία κοινή, χωρίς να αποδίδεται κάποια αξιολογική κρίση για το πρόσωπό του.",Χρυσόστομος
7,ΕΙΣΑΓΩΓΗ,Ξενοφώντα,[],"[Ψ Ξενοφώντα, Αθην. Πολιτ., 8 . Μάς λέει λοιπόν καθαρά ο άγνωστός μας ολιγαρχικός που έγραψε το φυλλάδιο για να κατηγορήσει το αθηναϊκό πολίτευμα, πως η αθηναϊκή γλώσσα έχασε την πρώτη της υφή και εξελίσσεται σε μια κοσμοπολιτική γλώσσα, γιατί δέχτηκε όχι μόνο λέξεις από τις άλλες ελληνικές ντοπιολαλιές, μα και βαρβαρικές.]",ουδέτερη,"Η αναφορά στον Ξενοφώντα είναι ουδέτερη, καθώς χρησιμοποιείται μόνο ως πηγή για ιστορική πληροφορία χωρίς αξιολογική κρίση για το πρόσωπό του.",Ξενοφών
8,ΚΕΦΑΛΑΙΟ ΠΡΩΤΟ,Βλαντής,[],[δικαιολογεί την έκδοση της Φυσικής του],ουδέτερη,"Αναφέρεται μόνο ως μεταφραστής και συντάκτης σχολικού εγχειριδίου, χωρίς αξιολογικούς χαρακτηρισμούς.",Βλαντής
9,ΚΕΦΑΛΑΙΟ ΠΡΩΤΟ,Καβαλιώτη,[ήξερε και τα ελληνικά καλά και τα βλάχικα και τα αρβανίτικα],[Σύνταξε λοιπόν γύρω στα 1770 στη Μοσχόπολη όπου ήταν δάσκαλος στην ανώτερη σχολή της ένα τρίγλωσσο λεξικό],θετική,"Περιγράφεται ως πολύγλωσσος και δημιουργικός, με θετική αναφορά στη συμβολή του στη λεξικογραφία και την εκπα

In [ ]:
# save
combined_df.to_csv('kordatos_quotes_framing.csv', index=False, encoding='utf-8')  
